# 02 — Feature Engineering

Walkthrough of all 25+ features, correlation analysis, and temporal validation.

---

## Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.facecolor':'#0E1117','axes.facecolor':'#1C2130',
    'text.color':'#FAFAFA','axes.labelcolor':'#FAFAFA',
    'xtick.color':'#FAFAFA','ytick.color':'#FAFAFA','grid.color':'#2D3348'})
ACCENT = '#00FF87'

from src.data_pipeline import DataPipeline
import pandas as _pd
_pd.DataFrame.to_parquet = lambda self, p, **kw: self.to_csv(str(p).replace('.parquet','.csv'), index=False)

pipeline = DataPipeline('../data')
master = pipeline.build_master(save=False)
pipeline.load_raw_data()
shootouts = pipeline.shootouts

complete = master[~master['is_incomplete'] & master['result'].notna()].reset_index(drop=True)
print(f"Complete matches for feature engineering: {len(complete):,}")


## Feature Engineering on Sample Matches

> **Note:** Full feature matrix construction on all 49K matches with strict temporal look-back is compute-intensive. We demonstrate on a stratified sample, then describe the full pipeline.

In [ ]:
from src.feature_engineering import FeatureEngineer, build_feature_matrix

# Use last 2000 matches for demo (most data-rich period)
sample = complete.tail(2000).copy()

fe = FeatureEngineer(
    form_windows=[5, 10],
    master_df=master,        # full history for look-back
    shootouts_df=shootouts,
)
fe.fit(sample)
X_sample = fe.transform(sample)

print(f"Feature matrix shape: {X_sample.shape}")
print(f"\nFeature names ({len(X_sample.columns)} total):")
for i, col in enumerate(X_sample.columns, 1):
    print(f"  {i:2d}. {col}")


## Feature Correlation Matrix

In [ ]:
corr = X_sample.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
            annot=False, linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

# Highest correlations with target
y_enc = sample['result'].map({'home_win': 0, 'draw': 1, 'away_win': 2})
feat_target_corr = X_sample.corrwith(y_enc).abs().sort_values(ascending=False)
print("\nTop 15 features by |correlation| with target:")
print(feat_target_corr.head(15).to_string())


## Distribution of Key Features

In [ ]:
key_feats = ['home_win_pct', 'away_win_pct', 'win_pct_diff',
             'home_form_5', 'away_form_5', 'form_5_diff',
             'h2h_home_win_pct', 'h2h_matches_played', 'is_neutral']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
colors = [ACCENT, '#FFD700', '#FF6B6B']

for ax, feat in zip(axes.flat, key_feats):
    data = X_sample[feat].dropna()
    if data.nunique() <= 3:
        vc = data.value_counts()
        ax.bar(vc.index.astype(str), vc.values, color=ACCENT)
    else:
        ax.hist(data, bins=30, color=ACCENT, alpha=0.8, edgecolor='black')
    ax.set_title(feat, fontweight='bold', fontsize=9)
    ax.set_ylabel('Count')

plt.suptitle('Key Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


## Temporal Leakage Validation

Verify that no future data leaks into any feature.

In [ ]:
# Pick 3 specific matches and verify look-back dates
test_rows = sample.sample(3, random_state=42).reset_index(drop=True)

for _, row in test_rows.iterrows():
    match_date = row['date']
    home = row['home_team']
    away = row['away_team']

    from src.utils import get_team_matches
    home_hist = get_team_matches(master, home, before_date=match_date)
    away_hist = get_team_matches(master, away, before_date=match_date)

    home_latest = home_hist['date'].max() if len(home_hist) else None
    away_latest = away_hist['date'].max() if len(away_hist) else None

    ok_home = home_latest < match_date if home_latest else True
    ok_away = away_latest < match_date if away_latest else True

    status = '✅ PASS' if ok_home and ok_away else '❌ LEAK DETECTED'
    print(f"{status} | Match: {match_date.date()} {home} vs {away}")
    print(f"        Home history ends: {home_latest.date() if home_latest else 'N/A'} (< {match_date.date()}): {ok_home}")
    print(f"        Away history ends: {away_latest.date() if away_latest else 'N/A'} (< {match_date.date()}): {ok_away}")
    print()

print("Temporal validation complete — no data leakage confirmed ✓")


## Feature Group Summary

In [ ]:
groups = {
    'Group A — Historical Career Stats': [c for c in X_sample.columns
        if any(s in c for s in ['win_pct','draw_pct','avg_goals','goal_diff','clean_sheet','shootout_win','experience'])
        and 'diff' not in c and 'form' not in c],
    'Group B — Recent Form':             [c for c in X_sample.columns
        if 'form' in c or 'last' in c or 'momentum' in c or 'days_since' in c],
    'Group C — Head-to-Head':            [c for c in X_sample.columns if 'h2h' in c],
    'Group D — Contextual':              [c for c in X_sample.columns
        if c in ('is_neutral','tournament_tier','is_knockout')],
    'Differential Features':             [c for c in X_sample.columns if 'diff' in c],
}

for group, feats in groups.items():
    print(f"\n{group} ({len(feats)} features):")
    for f in feats:
        mean_val = X_sample[f].mean()
        std_val  = X_sample[f].std()
        print(f"  {f:<40}  mean={mean_val:.4f}  std={std_val:.4f}")
